# Regularization : Ridge vs. Lasso 🥊🥊
Ridge and Lasso are two variations of the regularization technique. The difference comes from a slight change in the mathematical expression of the cost function used to train the model. In this exercise, these two regularization srtategies will be compared in order to analyze their respective influence on the model's coefficients. The effect of the regularization strength $\alpha$ will also be studied.

## Importing libraries and loading the dataset

1. Import the usual libraries. Don't forget to import scikit-learn's following model classes:
* LinearRegression
* Ridge
* Lasso

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

2. Execute the following line of code to read the dataset directly from our S3

In [2]:
dataset = pd.read_csv("https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/R%C3%A9gression+r%C3%A9gularis%C3%A9es/gene+data/data_lasso.csv",)

In [3]:
dataset.head()

,Unnamed: 0.1,Unnamed: 0,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,target
0,0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0,12.408154
1,1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0,13.414970
2,2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0,13.566183
3,3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0,12.943886
4,4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0,91.307146


3. The two first columns are useless, find a way to get rid of them.

In [4]:
dataset = dataset.iloc[:, 2:]
dataset.head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,target
0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,0.0,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0,12.408154
1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,0.0,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0,13.414970
2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,0.0,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0,13.566183
3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,0.0,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0,12.943886
4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,0.0,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0,91.307146


## Exploration

4. Print some information about the dataset. Do you notice something unusual ?

In [5]:
dataset.shape

(801, 20532)

5. Make sure there are no missing values at all in the whole dataset

In [6]:
dataset.isnull().values.any()

False

## Preparing the data for machine learning
6. Separate the target from the features

In [7]:
target = "target"

Y = dataset[target]
X = dataset[dataset.columns[dataset.columns != target]]
X

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.000000
1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.000000
2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.000000
3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.000000
4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,0.0,1.865642,2.718197,7.350099,10.006003,0.0,6.764792,0.496922,0.0,0.0,...,6.088133,9.118313,10.004852,4.484415,9.614701,12.031267,9.813063,10.092770,8.819269,0.000000
797,0.0,3.942955,4.453807,6.346597,10.056868,0.0,7.320331,0.000000,0.0,0.0,...,6.371876,9.623335,9.823921,6.555327,9.064002,11.633422,10.317266,8.745983,9.659081,0.000000
798,0.0,3.249582,3.707492,8.185901,9.504082,0.0,7.536589,1.811101,0.0,0.0,...,5.719386,8.610704,10.485517,3.589763,9.350636,12.180944,10.681194,9.466711,4.677458,0.586693
799,0.0,2.590339,2.787976,7.318624,9.987136,0.0,9.213464,0.000000,0.0,0.0,...,5.785237,8.605387,11.004677,4.745888,9.626383,11.198279,10.335513,10.400581,5.718751,0.000000


7. Make a train/test splitting

In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### Preprocessing
8. What preprocessings are necessary here ? Apply them to the features

In [9]:
pd.set_option('display.max_columns', None)
dataset.dtypes

dataset.dtypes.value_counts()


float64    20532
Name: count, dtype: int64

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train

array([[-0.18260929,  1.90922841,  0.55666368, ...,  2.56930826,
        -0.08594483, -0.26295175],
       [-0.18260929,  0.83064273,  0.28377528, ..., -0.66063474,
         2.33028951, -0.26295175],
       [-0.18260929,  0.49373092,  0.78493894, ..., -0.19430024,
         1.20339421, -0.26295175],
       ...,
       [-0.18260929, -0.38215881, -0.42249159, ..., -0.53615319,
        -0.65695452, -0.26295175],
       [-0.18260929,  0.96610584,  0.61747234, ...,  0.80191873,
         0.4136774 , -0.26295175],
       [-0.18260929,  0.00581922, -0.54792011, ...,  1.07668602,
         0.68093856, -0.26295175]])

In [11]:
X_test = scaler.transform(X_test)
X_test

array([[-0.18260929,  0.2429336 ,  1.37848035, ...,  0.39936478,
        -1.16087917, -0.26295175],
       [-0.18260929,  0.70568734,  0.21899154, ...,  0.76452297,
        -0.07575806, -0.26295175],
       [-0.18260929,  1.20507988,  0.45969636, ..., -0.77359771,
        -0.48715095, -0.26295175],
       ...,
       [-0.18260929,  0.43581279,  1.02260814, ..., -1.17520777,
        -0.55258271, -0.26295175],
       [-0.18260929, -0.3833982 , -0.38468189, ...,  0.36282142,
        -0.01442762, -0.26295175],
       [-0.18260929,  0.14835308, -0.37323333, ..., -0.9080833 ,
        -0.06517458, -0.26295175]])

## Ridge
Let's focus on Ridge regularization. We'll train 3 Ridge regressors with different values of the strength $\alpha$, and analyze the performances as well as the influence on the model's coefficients.

9. Declare an instance of the Ridge class with $\alpha = 1$. Save this instance for later analysis into an object called `ridge1`. Train the model and display its R2-score on train and test sets.

In [12]:
ridge1 = Ridge()
ridge1.fit(X_train, Y_train)

print("R2 score on training set : ", ridge1.score(X_train, Y_train))
print("R2 score on test set : ", ridge1.score(X_test, Y_test))

R2 score on training set :  0.9999999996577973
R2 score on test set :  0.9836218646691628


10. Declare an instance of the Ridge class with $\alpha = 1000000$. Save this instance for later analysis into an object called `ridge2`. Train the model and display its R2-score on train and test sets.

In [13]:
ridge2 = Ridge(alpha=1000000)
ridge2.fit(X_train, Y_train)

print("R2 score on training set : ", ridge2.score(X_train, Y_train))
print("R2 score on test set : ", ridge2.score(X_test, Y_test))

R2 score on training set :  0.6456028421883675
R2 score on test set :  0.6210598969099537


11. Declare an instance of the Ridge class with $\alpha=100000000$. Save this instance for later analysis into an object called `ridge3`
. Train the model and display its R2-score on train and test sets.

In [14]:
ridge3 = Ridge(alpha=100000000)
ridge3.fit(X_train, Y_train)

print("R2 score on training set : ", ridge3.score(X_train, Y_train))
print("R2 score on test set : ", ridge3.score(X_test, Y_test))

R2 score on training set :  0.015667456835194216
R2 score on test set :  0.01493565740552194


12. How do the scores vary when alpha changes? Can you explain what's happening?

13. Extract the coefficients of each of the three Ridge models, and store them into a DataFrame

In [15]:
features = X.columns

ridge_coef = pd.DataFrame({
    "Feature": features,
    "Ridge1": ridge1.coef_,
    "Ridge2": ridge2.coef_,
    "Ridge3": ridge3.coef_
})

ridge_coef.head()

,Feature,Ridge1,Ridge2,Ridge3
0,gene_0,-0.008681,-0.001182,-0.000021
1,gene_1,-0.014291,-0.002421,-0.000029
2,gene_2,-0.010445,-0.003514,-0.000053
3,gene_3,-0.006097,-0.000736,-0.000008
4,gene_4,0.005259,0.004188,0.000077


14. Plot the coefficients of the three models in the same figure. What do you notice?

In [16]:
import plotly.express as px
custom_colors = {
    'Ridge1': "#40adb5",
    'Ridge2': "#3d7ca8",
    'Ridge3': "#0a3656",
}

fig = px.line(
    ridge_coef,
    x="Feature",
    y=["Ridge1", "Ridge2", "Ridge3"],
    title="Ridge Regression Coefficients",
    labels={"value": "Coefficient", "variable": "Ridge Regression Model"},
    color_discrete_map=custom_colors
)

fig.show()

## Lasso
Let's make the same study with Lasso regularization.

15. Declare an instance of the Lasso class with $\alpha = 1$. Save this instance for later analysis into an object called `lasso1`. Train the model and display its R2-score on train and test sets.

In [17]:
lasso1 = Lasso(alpha=1)
lasso1.fit(X_train, Y_train)

print("R2 score on training set : ", lasso1.score(X_train, Y_train))
print("R2 score on test set : ", lasso1.score(X_test, Y_test))

R2 score on training set :  0.9869991549190295
R2 score on test set :  0.9795379358283144


16. Declare an instance of the Lasso class with $\alpha = 30$. Save this instance for later analysis into an object called `lasso2`. Train the model and display its R2-score on train and test sets.

In [18]:
lasso2 = Lasso(alpha=30)
lasso2.fit(X_train, Y_train)

print("R2 score on training set : ", lasso2.score(X_train, Y_train))
print("R2 score on test set : ", lasso2.score(X_test, Y_test))

R2 score on training set :  0.2094242867325934
R2 score on test set :  0.2098146140156445


17. Declare an instance of the Lasso class with $\alpha = 100$. Save this instance for later analysis into an object called `lasso3`. Train the model and display its R2-score on train and test sets.

In [19]:
lasso3 = Lasso(alpha=100)
lasso3.fit(X_train, Y_train)

print("R2 score on training set : ", lasso3.score(X_train, Y_train))
print("R2 score on test set : ", lasso3.score(X_test, Y_test))

R2 score on training set :  0.0
R2 score on test set :  -1.619771758631927e-05


18. Plot the coefficients of the three models in the same figure. What do you notice?

In [20]:
features = X.columns

lasso_coef = pd.DataFrame({
    "Feature": features,
    "Lasso1": lasso1.coef_,
    "Lasso2": lasso2.coef_,
    "Lasso3": lasso3.coef_
})
lasso_coef.head()

,Feature,Lasso1,Lasso2,Lasso3
0,gene_0,-0.0,-0.0,-0.0
1,gene_1,-0.0,-0.0,-0.0
2,gene_2,-0.0,-0.0,-0.0
3,gene_3,0.0,-0.0,-0.0
4,gene_4,0.0,0.0,0.0


In [21]:
fig =  px.line(
    lasso_coef,
    x="Feature",
    y=["Lasso1", "Lasso2", "Lasso3"],
    title="Lasso Regression Coefficients",
    labels={"value": "Coefficient", "variable": "Lasso Regression Model"},
)

fig.show()

## Hyperparameter optimization
19. Use grid search to find the best value for $\alpha$, for Ridge and then for Lasso. You can test the following list of values:
* Ridge: $\alpha = $ [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]
* Lasso: $\alpha = $ [1, 2, 3, 5, 10, 20, 30]

In [22]:
ridge_params = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100],
}
ridge = Ridge()

ridge_grid_search = GridSearchCV(estimator=ridge, param_grid=ridge_params, cv=5, scoring='r2')

ridge_grid_search.fit(X_train, Y_train)

print("Meilleur alpha :", ridge_grid_search.best_params_)
print("Meilleur score R² :", ridge_grid_search.best_score_)

Meilleur alpha : {'alpha': 0.01}
Meilleur score R² : 0.9830659570819469


In [23]:
lasso_params = {
    'alpha': [1, 2, 3, 5, 10, 20, 30],
}
lasso = Lasso()
lasso_grid_search = GridSearchCV(estimator=lasso, param_grid=lasso_params, cv=5, scoring='r2')
lasso_grid_search.fit(X_train, Y_train)
print("Meilleur alpha :", lasso_grid_search.best_params_)
print("Meilleur score R² :", lasso_grid_search.best_score_)

Meilleur alpha : {'alpha': 1}
Meilleur score R² : 0.9808962374072732


### Comparing models
20. Display the scores on train set and test set for the best Ridge and for the best Lasso model.

In [24]:
ridge_best = Ridge(alpha=0.1)
ridge_best.fit(X_train, Y_train)
print("R2 score on training set with best alpha : ", ridge_best.score(X_train, Y_train))
print("R2 score on test set with best alpha : ", ridge_best.score(X_test, Y_test))

R2 score on training set with best alpha :  0.9999999999965764
R2 score on test set with best alpha :  0.983622342502


In [25]:
lasso_best = Lasso(alpha=1)
lasso_best.fit(X_train, Y_train)
print("R2 score on training set with best alpha : ", lasso_best.score(X_train, Y_train))
print("R2 score on test set with best alpha : ", lasso_best.score(X_test, Y_test))

R2 score on training set with best alpha :  0.9869991549190295
R2 score on test set with best alpha :  0.9795379358283144


21. Plot the coefficients of the best Ridge and best Lasso models in the same figure. If you had to deploy a model in production, which one would you choose and why?

In [26]:
features = X.columns
ridge_best_coef = pd.DataFrame({
    "Feature": features,
    "Best_Ridge": ridge_best.coef_,
    "Best_Lasso": lasso_best.coef_
})
ridge_best_coef.head()

,Feature,Best_Ridge,Best_Lasso
0,gene_0,-0.008682,-0.0
1,gene_1,-0.014297,-0.0
2,gene_2,-0.010446,-0.0
3,gene_3,-0.006097,0.0
4,gene_4,0.005259,0.0


In [27]:
fig = px.line(
    ridge_best_coef,
    x="Feature",
    y=["Best_Ridge", "Best_Lasso"],
    title="Best Ridge and Lasso Regression Coefficients",
    labels={"value": "Coefficient", "variable": "Regression Model"},
    color_discrete_map={"Best_Ridge": "#40adb5", "Best_Lasso": "#3d7ca8"}
)
fig.show()

## Using Lasso as an automated feature selection method
Let's focus on the best Lasso model. Only a few features have non-zero coefficients, which means that we could train a model by including only these features, and still get the same performances.

22. Extract the names of the features that have non-zero coefficients in Lasso.

In [28]:
mask = ridge_best_coef['Best_Lasso'] != 0
best_features = ridge_best_coef.loc[mask, 'Feature'].to_list()
best_features

['gene_221',
 'gene_357',
 'gene_747',
 'gene_864',
 'gene_1063',
 'gene_1384',
 'gene_2073',
 'gene_2655',
 'gene_2747',
 'gene_2910',
 'gene_3569',
 'gene_4481',
 'gene_4773',
 'gene_5027',
 'gene_5407',
 'gene_5523',
 'gene_5578',
 'gene_5815',
 'gene_5934',
 'gene_6611',
 'gene_6748',
 'gene_6795',
 'gene_6876',
 'gene_6940',
 'gene_7623',
 'gene_7964',
 'gene_8027',
 'gene_8032',
 'gene_8594',
 'gene_8598',
 'gene_8729',
 'gene_8739',
 'gene_9652',
 'gene_10106',
 'gene_10357',
 'gene_10548',
 'gene_10707',
 'gene_11346',
 'gene_11412',
 'gene_11920',
 'gene_12167',
 'gene_12194',
 'gene_12866',
 'gene_13103',
 'gene_13147',
 'gene_13928',
 'gene_14062',
 'gene_14092',
 'gene_15306',
 'gene_15589',
 'gene_15830',
 'gene_16215',
 'gene_16372',
 'gene_16385',
 'gene_16557',
 'gene_17012',
 'gene_17018',
 'gene_17317',
 'gene_17354',
 'gene_17801',
 'gene_17847',
 'gene_17892',
 'gene_17905',
 'gene_17947',
 'gene_18631',
 'gene_18746']

23. Filter the feature matrix X to keep only these features, make a new train/test splitting from this X matrix and do the preprocessing once again

In [29]:
X = X.loc[:, best_features]
X.head()

,gene_221,gene_357,gene_747,gene_864,gene_1063,gene_1384,gene_2073,gene_2655,gene_2747,gene_2910,gene_3569,gene_4481,gene_4773,gene_5027,gene_5407,gene_5523,gene_5578,gene_5815,gene_5934,gene_6611,gene_6748,gene_6795,gene_6876,gene_6940,gene_7623,gene_7964,gene_8027,gene_8032,gene_8594,gene_8598,gene_8729,gene_8739,gene_9652,gene_10106,gene_10357,gene_10548,gene_10707,gene_11346,gene_11412,gene_11920,gene_12167,gene_12194,gene_12866,gene_13103,gene_13147,gene_13928,gene_14062,gene_14092,gene_15306,gene_15589,gene_15830,gene_16215,gene_16372,gene_16385,gene_16557,gene_17012,gene_17018,gene_17317,gene_17354,gene_17801,gene_17847,gene_17892,gene_17905,gene_17947,gene_18631,gene_18746
0,10.007966,0.000000,9.467555,0.000000,5.634797,6.909077,7.025970,9.148459,3.877077,2.717803,8.282532,4.974043,9.840658,0.000000,7.974615,8.518295,2.185898,3.877077,8.338077,8.450894,8.835839,3.926037,8.440400,13.221398,5.663951,5.248778,0.000000,1.822037,1.010279,7.210184,5.323766,8.347608,5.748037,1.010279,3.340391,11.986820,3.266292,2.602077,8.514300,6.896841,6.403961,7.748327,6.915150,11.422175,4.801122,6.795105,9.373618,5.663951,9.456143,0.00000,8.003417,10.898541,10.314527,9.113101,9.968854,10.841863,3.105561,7.627724,10.543564,8.803453,3.105561,0.0,0.000000,12.365904,8.034441,7.689285
1,6.375557,0.000000,6.997394,1.004394,8.882967,6.058490,8.614231,8.858941,2.530820,0.000000,6.799851,2.530820,0.000000,0.323658,3.137061,7.717813,0.587845,3.866146,9.401078,6.178577,10.269092,5.815972,6.484409,7.993159,0.000000,3.890826,0.000000,1.004394,2.328951,2.762370,2.814879,4.768274,1.465034,0.000000,1.004394,9.550341,5.064008,4.293724,3.708540,0.000000,3.890826,8.309099,7.508571,1.813607,2.914488,5.095979,9.685443,3.841047,10.348341,0.00000,9.320507,10.983364,6.451617,9.542481,9.624850,10.189923,0.587845,5.330483,10.294173,5.374883,0.323658,0.0,0.000000,14.094201,7.101471,8.292989
2,7.716408,0.452595,6.179666,0.000000,5.213829,11.596180,7.202535,8.390565,0.452595,0.000000,9.650298,3.065210,10.485527,0.000000,4.947105,0.452595,0.000000,1.306846,8.649289,3.186960,10.629065,4.306933,3.352236,13.047345,1.306846,1.074163,3.634977,4.981168,0.000000,3.677147,1.683023,7.060091,2.533663,1.839758,3.000252,11.916812,2.866690,11.369221,4.009723,8.497401,5.094422,7.468746,3.244187,0.000000,1.507160,6.464881,8.848247,5.726766,9.644994,0.00000,8.504736,10.389717,9.306825,11.194683,9.898288,9.708642,3.500522,2.337254,10.946307,1.074163,3.065210,0.0,0.000000,11.461280,6.365222,10.833428
3,8.309085,0.000000,9.121088,4.737725,5.535459,12.718734,8.331741,9.807848,2.175652,1.267356,8.110885,4.321690,10.071087,0.000000,7.073392,8.906247,0.000000,4.598014,8.806218,5.942700,9.199147,4.489235,5.967242,13.604530,0.000000,2.650029,1.039419,1.267356,0.434882,7.226961,2.650029,7.809479,2.566840,4.679497,5.134369,12.555325,6.002820,5.524474,6.007245,7.992321,5.883750,6.434378,6.253292,6.589990,3.127534,3.573556,9.367646,6.343552,11.908640,0.00000,8.914687,10.184020,14.117408,10.302433,9.347533,10.865068,2.728704,5.866458,10.183586,5.785961,5.060251,0.0,0.000000,13.686227,7.092112,9.562801
4,3.370332,3.208502,11.227267,0.000000,7.181162,3.335812,9.977938,9.898096,8.711374,10.517246,11.873982,0.649386,0.000000,2.396187,12.870781,2.678342,6.416654,8.837760,9.111107,3.452793,6.741925,4.836651,13.636772,9.304625,5.024776,0.000000,10.271988,8.453908,7.408185,10.982102,7.912446,9.180951,10.828644,0.000000,1.435949,4.461804,1.942120,2.544139,0.889707,1.711142,4.345574,6.783470,10.565407,3.525831,2.801097,4.283751,9.063794,14.189315,9.379807,14.97592,10.729952,8.747129,6.291349,10.597187,9.745926,11.752414,7.939432,9.407247,9.273817,1.435949,6.322429,0.0,12.010059,9.842449,9.854462,14.439409


In [30]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [31]:
# Create scaler for numeric features
scaler = StandardScaler()

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = scaler.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = scaler.transform(X_test) # Don't fit again !!
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
      gene_221   gene_357   gene_747  gene_864  gene_1063  gene_1384  \
364   9.598170   0.000000   7.659989  2.125056   8.459862   6.481825   
458   9.221942   0.000000   9.757558  1.594978   8.673161   5.130033   
76    6.082534  10.392221  10.898480  0.000000   7.825843   5.664523   
64   12.108841   0.444349   8.225516  0.783582   6.487625   7.464913   
638   5.269875  10.237043  12.303344  0.000000   6.024218   3.375470   

     gene_2073  gene_2655  gene_2747  gene_2910  gene_3569  gene_4481  \
364   6.615035   9.405637   3.269856   0.000000  10.862041   8.628675   
458   6.724500   8.778718   2.951644   0.000000  10.361856   1.747473   
76    9.570061   9.050831   6.818557   5.426466  11.128014   0.000000   
64    7.161112   9.186951   2.508251   1.058109  10.101529   5.559856   
638   8.899227  10.065632  11.303256   5.792980  11.749015   0.333996   

     gene_4773  gene_5027  gene_5407  gene_5523  gene_5578  gene_5815  \
364   0.00000

24. Train a NON-regularized linear regression and evaluate the performances. What do you think of the result?

In [33]:
# Train linear regression (no regularization)
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


In [34]:
# Print R^2 scores
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.9910017735998236
R2 score on test set :  0.989466602650973
